# Manage Feedback

#### Imports

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

# Add the parent directory (or another appropriate path) to sys.path so Python can find Exclusion_functions
notebook_dir = os.path.dirname(os.path.abspath('04_Optimization.ipynb'))
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..', '..', '..'))
function_dir = os.path.abspath(os.path.join(parent_dir, 'Function_Files'))
if function_dir not in sys.path:
    sys.path.append(function_dir)

import pandas as pd
from openai_api import OpenAIAgent
import nest_asyncio
nest_asyncio.apply()
from IPython.display import Markdown, display
def printmd(string):
    display(Markdown(string))
    
import Feedback_functions as ff
import Matching_functions as mf
import datetime

#### Variables

In [ ]:
CATEGORY = "Cups"
today = datetime.datetime.now().strftime('%m.%d.%Y')
ip_path = f"{parent_dir}\\Data\\"
ip_path_cat = f"{parent_dir}\\Data\\{CATEGORY}\\"
OP_PATH = f"{parent_dir}\\Data\\{CATEGORY}\\Output\\"

In [ ]:
im_final = pd.read_csv(f"{OP_PATH}{CATEGORY}_matches.csv")

#### Read in Feedback

In [ ]:
im_final_with_feedback = ff.consolidate_feedback_from_excel([f"{ip_path_cat}Cups_Subs_9.2.2025_Review.xlsx"], im_final)

In [ ]:
im_final_with_feedback

#### Evaluate Results

In [ ]:
ff.get_feedback_summary(im_final_with_feedback)

In [ ]:
ff.get_reviewed_coverage(im_final, im_final_with_feedback)

In [ ]:
im_final_with_feedback['Correct'] = im_final_with_feedback['Correct'].replace({'Accept': 'Match'})
im_final_with_feedback['Correct'] = im_final_with_feedback['Correct'].replace({'Consider': 'Match'})
im_final_with_feedback['Correct'] = im_final_with_feedback['Correct'].replace({'Reject': 'No Match'})

#### Get New Prompts/Matches

In [ ]:
user_prompts, numerical_id_to_entity_id_map = ff.user_prompts_with_rules(im_final, im_final_with_feedback, use_subcategory_rules = False)

In [ ]:
for user_prompt in user_prompts:
    print(user_prompt)
    print('----'*40)


In [ ]:
im_final = mf.generate_matches(im_final, user_prompts,numerical_id_to_entity_id_map, hard_rules = "If mentioned, size/volume should be a match if not very close. Same with color, shape, etc. You should not be trying to swap a 20 oz product with a 16 oz product.", batch_model= False)

#### Adjust New Matches From Feedback

In [ ]:
im_final2 = ff.update_matches_based_on_feedback_sym(im_final, im_final_with_feedback, feedback_correct_col = 'Correct', accept_value = "Match", reject_value='No Match')

#### Write new matches

In [ ]:
mf.write_top_matches(im_final2, f"{OP_PATH}{CATEGORY}_Subs_{today}.xlsx", n = 35, pl = True, enable_vpn_exclusion = True)

#### Save files

In [ ]:
im_final2.to_csv(f"{OP_PATH}{CATEGORY}_matches_post_feedback_1.csv", index=False)
im_final_with_feedback.to_csv(f"{OP_PATH}{CATEGORY}_feedback_df.csv", index=False)